#### AIS Maritime SQL Agent (v1.4) — ReAct + Sliding-Window Memory

This version keeps the same **ReAct-style “agent owns synthesis”** loop and the same **validate → execute → heal** guardrails, but upgrades the runtime stability by adding **message pruning** so the agent can run longer sessions without ballooning context or breaking tool-call pairing.

#### Key Upgrades vs v1.3
* **Sliding-Window Memory (Message Trimming):** Before calling the LLM, the agent now prunes the conversation with `trim_messages(...)`, keeping only the most recent window to reduce context growth and prevent long-session degradation.
* **Tool-Call Pair Safety (`allow_partial=False`):** Trimming is configured to never split an `AIMessage` tool-call from its corresponding `ToolMessage`, avoiding tool history mismatches and API/tool-call crash edge cases.
* **Conversation Starts on User (`start_on="human"`):** The pruned window is forced to begin with a `HumanMessage`, keeping the LLM input well-formed (user → assistant) even after heavy trimming.
* **No Logic Changes to Guardrails:** Validation, execution, fixer retries, and tool-call routing remain the same; the main difference is *what subset of `messages` is passed into the agent prompt*.

#### Updated Execution Flow (ReAct Loop)
0. **Context Load:** `messages` are persisted via `MemorySaver` for session continuity.
1. **Prune + Reason + Decide:** `agent_node`
   * Trims history into `pruned_messages` (safe sliding window).
   * Produces either:
     * A conversational answer (no tool call) → `END`
     * A SQL tool call (`execute_sql`) → `validator_node`
2. **Validate:** `validator_node`
   * *If safe:* → `executor_node`
   * *If invalid:* → `fixer_node`
3. **Execute:** `executor_node`
   * *If success:* Returns a `ToolMessage` (linked via `tool_call_id`) → loops back to `agent_node`
   * *If DB/tool error:* Sets critique → `fixer_node`
4. **Heal:** `fixer_node`
   * Rewrites SQL → returns to `validator_node`
   * After max retries: emits a `ToolMessage` error → `agent_node`
5. **Respond:** `agent_node` synthesizes the final response using ToolMessage observations → `END`

In [8]:
import os
from typing import TypedDict, Literal, Annotated
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_core.messages import trim_messages

from dotenv import load_dotenv
load_dotenv()

# --- PHASE 1: DATABASE CONNECTIVITY ---

def get_database_connection():
    db_user = os.getenv("DB_USER")
    db_password = os.getenv("DB_PASSWORD")
    db_host = os.getenv("DB_HOST")
    db_port = os.getenv("DB_PORT")
    db_name = os.getenv("DB_NAME")

    db_uri = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
    
    return SQLDatabase.from_uri(
        db_uri,
        include_tables=['ais_data'],
        sample_rows_in_table_info=2
    )

db = get_database_connection()

# --- PHASE 2: TOOL DEFINITION ---

@tool
def execute_sql(query: str) -> str:
    """Executes a PostgreSQL query against the ais_data table and returns the result."""
    try:
        return str(db.run(query))
    except Exception as e:
        return f"ERROR: {str(e)}"

# --- PHASE 3: STATE MANAGEMENT ---

class AgentState(TypedDict, total=False):
    # 'messages' now tracks the full ReAct flow: Human -> AI (ToolCall) -> ToolMessage -> AI (Final Answer)
    messages: Annotated[list, add_messages]  
    schema_context: str          
    sql_query: str               
    current_tool_call_id: str    # NEW: Tracks the ID of the tool call for the ToolMessage
    validation_status: str       
    critique: str                
    retry_count: int         

# --- PHASE 4: NODE DEVELOPMENT ---

MODEL = "openai/gpt-oss-120b"
BASE_URL = "https://api.groq.com/openai/v1"

llm = ChatOpenAI(
    model=MODEL,
    base_url=BASE_URL,
    api_key=os.environ["API_KEY"],
    temperature=0.3,
)

AIS_SCHEMA_INFO = """
Table: ais_data

# COLUMN DEFINITIONS
- mmsi (text): Unique vessel identifier
- basedatetime (timestamp): Time of position report (Format: 'YYYY-MM-DDTHH:MM:SS')
- lat (float): Latitude 
- lon (float): Longitude 
- sog (float): Speed over ground in knots 
- cog (float): Course over ground in degrees 
- heading (float): True heading in degrees (0 to 359). The value 511 means 'Not Available'. 
- vesselname (text): Name of the ship (Always uppercase)
- imo (text): IMO number (Always starts with 'IMO')
- callsign (text): Call sign 
- vesseltype (integer): Numeric ITU-R M.1371 code for vessel category. 
- status (integer): Numeric ITU-R M.1371 code for navigation status. 
- length (float): Vessel length in meters 
- width (float): Vessel width in meters 
- draft (float): Vessel draft in meters 
- cargo (integer): Numeric code for cargo type.
- transceiverclass (text): AIS class (Exactly 'A' or 'B')
- geometry (geometry): PostGIS geometry column (Point)

# STATUS CODES MAPPING (`status`)
0=Moving/Under way, 1=At anchor, 2=Not under command, 3=Restricted maneuverability, 4=Constrained by draught, 5=Moored/Docked, 6=Aground, 7=Fishing, 8=Sailing, 11=Towing astern, 12=Pushing ahead/towing alongside, 14=Search and Rescue active, 15=Undefined.

# VESSEL TYPE CODES MAPPING (`vesseltype`)
30=Fishing, 31=Towing, 32=Large Towing, 33=Dredging/Underwater ops, 34=Diving ops, 35=Military ops, 36=Sailing, 37=Pleasure Craft, 40-49=High-Speed Craft (HSC), 50=Pilot Vessel, 51=Search and Rescue, 52=Tugs, 53=Port Tenders, 54=Anti-pollution, 55=Law Enforcement, 58=Medical, 60-69=Passenger Ships, 70-79=Cargo Ships, 80-89=Tankers.

# CRITICAL TRANSLATION RULES
1. When users ask for specific types of vessels or statuses, map their natural language to the integer codes. 
   - Example: 'moving cargo ships' -> `vesseltype BETWEEN 70 AND 79 AND status = 0`.
   - Example: 'parked or docked tankers' -> `vesseltype BETWEEN 80 AND 89 AND status IN (1, 5)`.
   - Example: 'ships in distress or broken down' -> `status IN (2, 6, 14)`.
2. If calculating averages, minimums, or maximums for heading, OR if filtering for valid headings, you MUST exclude 511 (e.g., `heading != 511`).
3. THE "CURRENT STATE" RULE (TIME-SERIES DEDUPLICATION):
Because this is a time-series database, a single ship (mmsi) has many historical rows. Whenever a user asks for the "current" state, "present" location, or a general fleet-wide query (e.g., "Find all passenger ships", "How many tankers", "Show moving tugs"), you MUST deduplicate the data to get ONLY the most recent ping per vessel.
- ALWAYS use a CTE (Common Table Expression) with `DISTINCT ON (mmsi)` and `ORDER BY mmsi, basedatetime DESC`.
- Apply your `WHERE` filters (vesseltype, status, etc.) inside the CTE to optimize performance.
- Query your final results (SELECT columns, COUNT, AVG) from this CTE.

# PREDICTIVE ANALYTICS & MATH RULES
If a user asks where a ship WILL BE in a future time or hour (Projection), you MUST use PostGIS `ST_Project`.
- Distance math: `sog` is in knots. You must multiply `sog` by 0.514444 to get meters per second. Multiply that by the requested time in seconds.
- Azimuth math: `cog` is in degrees. You must convert it using `radians(cog)`.
- Always use the most recent ping by adding `ORDER BY basedatetime DESC LIMIT 1`.

# EXAMPLES (Few-Shot Prompting)
Q: "How many Class A tankers are currently moored?"
SQL: SELECT COUNT(*) FROM ais_data WHERE transceiverclass = 'A' AND vesseltype BETWEEN 80 AND 89 AND status = 5;

Q: "Where will the ship MISS CHRISTY be in 2 hours?"
SQL: SELECT ST_AsText(ST_Project(geometry::geography, (sog * 0.514444) * (2 * 3600), radians(cog))) AS predicted_location FROM ais_data WHERE vesselname = 'MISS CHRISTY' ORDER BY basedatetime DESC LIMIT 1;

Q: "Show me the names of 5 military ships that are currently moving."
SQL: SELECT vesselname FROM ais_data WHERE vesseltype = 35 AND status = 0 LIMIT 5;

Q: "Find the speed and heading of the ship named LEICESTER."
SQL: SELECT sog, heading FROM ais_data WHERE vesselname = 'LEICESTER';

Q: "What is the average heading of moving passenger ships?"
SQL: SELECT AVG(heading) FROM ais_data WHERE vesseltype BETWEEN 60 AND 69 AND status = 0 AND heading != 511;

Q: "How many Class A tankers are currently moored?"
SQL: 
WITH latest AS (
  SELECT DISTINCT ON (mmsi) mmsi 
  FROM ais_data 
  WHERE transceiverclass = 'A' AND vesseltype BETWEEN 80 AND 89 AND status = 5 
  ORDER BY mmsi, basedatetime DESC
)
SELECT COUNT(*) FROM latest;

Q: "Find the current speed and heading of the ship named LEICESTER."
SQL: 
SELECT sog, heading 
FROM ais_data 
WHERE vesselname = 'LEICESTER' 
ORDER BY basedatetime DESC LIMIT 1;
"""

VALIDATOR_SCHEMA_INFO = """
Table: ais_data
Columns: mmsi (text), basedatetime (timestamp), lat (float), lon (float), sog (float), cog (float), heading (float, ignore 511), vesselname (text), imo (text), callsign (text), vesseltype (int), status (int), length (float), width (float), draft (float), cargo (int), transceiverclass (text, 'A' or 'B'), geometry (PostGIS point).
Valid status codes: 0, 1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 14, 15.
Valid vesseltype codes: 30-37, 40-55, 58, 60-69 (Passenger), 70-79 (Cargo), 80-89 (Tankers).

SAFE FUNCTIONS ALLOWED:
- Standard Aggregates: COUNT, AVG, MIN, MAX
- PostGIS Spatial & Projection: ST_Project, ST_AsText, ST_Distance, ST_DWithin, ST_MakePoint, ST_MakeEnvelope
- Math/Casting: radians(), ::geography, basic multiplication/division.
"""

def agent_node(state: AgentState):
    """Node A: The Brain (Reasoning & Synthesis)"""
    print("\n--- AGENT THINKING ---")
    
    # --- MEMORY PRUNING (SLIDING WINDOW) ---
    # Keeps the last 10 messages in the array.
    # allow_partial=False ensures we never accidentally separate an AIMessage tool-call 
    # from its corresponding ToolMessage, preventing API crash errors.
    pruned_messages = trim_messages(
        state['messages'],
        max_tokens=25, 
        token_counter=len,
        strategy="last",
        allow_partial=False,
        start_on="human" # <--- NEW: Forces the array to always start with a user query
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert maritime AI assistant and PostGIS database analyst. 
        You have access to a PostgreSQL database containing AIS ship telemetry data.
        
        Database Schema & Translation Rules:
        {schema}
        
        CORE INSTRUCTIONS:
        
        1. WHEN TO USE THE DATABASE (TOOL USE): 
        If the user asks for specific data, statistics, vessel details, or spatial locations, you MUST use the `execute_sql` tool to generate and run a PostgreSQL query.
        - ALWAYS use 'ais_data' as the table name.
        - PROTECT THE DATABASE: Unless the user asks for an aggregation (like COUNT or AVG), always append a `LIMIT 10` (or whatever number the user specifies) to prevent massive data pulls.
        - SPATIAL AWARENESS: Use PostGIS functions (e.g., ST_DWithin, ST_MakeEnvelope) on the `geometry` column if the user asks for distance or bounding box queries.
        - NEVER hallucinate column names. Strictly adhere to the columns and definitions in the schema provided.
        - If the user asks a complex question requiring multiple pieces of information, you can use the `execute_sql` tool multiple times in sequence to gather all the data you need.

        2. WHEN TO CHAT (NO TOOL USE): 
        If the user says hello, asks general maritime knowledge, or explanation of a value from the AIS database (e.g., "What does AIS stand for?" or "What does the Vessel Status mean?"), or asks an unrelated question, DO NOT use the tool. Provide a helpful, natural language response directly.
        
        SYNTHESIS INSTRUCTION:
        1. If you had to use the DATABASE (TOOL USE), ONLY synthesize a final response AFTER you have successfully retrieved data using the `execute_sql` tool.
        2. If you didn't have to use the DATABASE TOOL, you may answer naturally without the tool.
        """),
        MessagesPlaceholder(variable_name="messages")
    ])
    
    llm_with_tools = llm.bind_tools([execute_sql])
    chain = prompt | llm_with_tools
    
    # We pass 'pruned_messages' to the LLM instead of the full 'state['messages']'
    response = chain.invoke({
        "schema": AIS_SCHEMA_INFO, 
        "messages": pruned_messages 
    })
    
    # Update state with the agent's new message
    state_update = {"messages": [response], "schema_context": AIS_SCHEMA_INFO}
    
    # Did the agent call the tool?
    if response.tool_calls:
        tool_call = response.tool_calls[0]
        state_update["sql_query"] = tool_call['args']['query']
        state_update["current_tool_call_id"] = tool_call['id']
        state_update["retry_count"] = 0 # Reset retries for a new query
        print(f"Decision: Tool call requested. \nQuery:\n {state_update['sql_query']}")
    else:
        print("Decision: Gathered enough context. Providing final answer.")
        
    return state_update

def validator_node(state: AgentState):
    """Node B: The Critic"""
    print("--- VALIDATING SQL ---")
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a strict PostgreSQL Security Validator.
        Review the following query for the `ais_data` table against this schema:
        {validation_schema}
        
        CRITICAL CHECKS:
        1. READ-ONLY: Reject if it contains DROP, DELETE, INSERT, UPDATE, ALTER.
        2. HALLUCINATIONS: Reject if it queries a column name NOT listed in the schema.
        3. DATA TYPES: Reject if `status` or `vesseltype` are queried as text strings instead of their valid integers.
        
        Output format: Return exactly "VALID" if safe. If invalid, return a concise explanation of the exact error."""),
        ("human", "Query to check: {query}")
    ])
    
    response = (prompt | llm).invoke({
        "query": state['sql_query'],
        "validation_schema": VALIDATOR_SCHEMA_INFO 
    })
    
    status = "VALID" if "VALID" in response.content.upper() else "INVALID"
    if status == "INVALID":
        print(f"Validation Failed: {response.content}")
    
    return {"validation_status": status, "critique": response.content if status == "INVALID" else ""}

def executor_node(state: AgentState):
    """Node C: The Executor"""
    print("--- EXECUTING SQL ---")
    try:
        result = execute_sql.invoke({"query": state['sql_query']})
        
        if isinstance(result, str) and result.startswith("ERROR:"):
            print("--- DATABASE ERROR CAUGHT ---")
            return {"critique": f"Database Execution Error: {result}", "validation_status": "INVALID"}
            
        print(f"Query successful. Result length: {len(result)}")
        # If successful, we pass the data back to the agent as a ToolMessage
        return {
            "messages": [ToolMessage(content=str(result), tool_call_id=state['current_tool_call_id'])],
            "critique": ""
        }
    except Exception as e:
        print("--- CRITICAL SYSTEM ERROR CAUGHT ---")
        return {"critique": f"System Invocation Error: {str(e)}", "validation_status": "INVALID"}

def fixer_node(state: AgentState):
    """Node D: The Repairman (Invisible to the main Agent loop)"""
    print("--- FIXING SQL ---")
    current_retries = state.get('retry_count', 0)
    
    # Prevent infinite fixing loops. If we fail 3 times, send an error back to the Agent.
    if current_retries >= 3:
        print("--- MAX RETRIES REACHED. RETURNING ERROR TO AGENT ---")
        return {
            "messages": [ToolMessage(content="ERROR: Failed to execute query due to persistent database errors. Notify the user.", tool_call_id=state['current_tool_call_id'])],
            "retry_count": current_retries + 1
        }
    
    # Extract the user's original intent from the message history to give the fixer context
    user_intent = next((msg.content for msg in reversed(state['messages']) if isinstance(msg, HumanMessage)), "Unknown Context")

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an Expert PostgreSQL Debugger. 
        Fix this failed query.
        Original Request: {question}
        Broken Query: {query}
        Critique: {critique}
        
        Schema Rules: {schema}
        Task: Rewrite the query permanently fixing the error. Use `execute_sql` tool."""),
    ])
    
    llm_with_tools = llm.bind_tools([execute_sql], tool_choice="required")
    response = (prompt | llm_with_tools).invoke({
        "question": user_intent,
        "query": state['sql_query'],
        "critique": state['critique'],
        "schema": state['schema_context']
    })
    
    fixed_sql = response.tool_calls[0]['args']['query']
    print(f"Proposed Fix: {fixed_sql}")
    
    return {"sql_query": fixed_sql, "retry_count": current_retries + 1}

# --- PHASE 5: GRAPH CONSTRUCTION & ROUTING ---

def route_agent_action(state: AgentState) -> Literal["validator", END]:
    """If the agent called a tool, go validate it. If it just output text, we are done."""
    last_message = state['messages'][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "validator"
    return END

def should_continue_validation(state: AgentState) -> Literal["executor", "fixer"]:
    return "executor" if state['validation_status'] == "VALID" else "fixer"

def should_continue_execution(state: AgentState) -> Literal["agent", "fixer"]:
    """If critique is empty, it means execution succeeded -> loop back to Agent."""
    return "agent" if not state.get("critique") else "fixer"

def should_continue_fixing(state: AgentState) -> Literal["validator", "agent"]:
    """If max retries hit, go back to Agent so it can synthesize an apology."""
    if state.get('retry_count', 0) > 3:
        return "agent" 
    return "validator"


workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("validator", validator_node)
workflow.add_node("executor", executor_node)
workflow.add_node("fixer", fixer_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", route_agent_action)
workflow.add_conditional_edges("validator", should_continue_validation)
workflow.add_conditional_edges("executor", should_continue_execution)
workflow.add_conditional_edges("fixer", should_continue_fixing)

# ==========================================
# COMPILATION & EXECUTION
# ==========================================
memory = MemorySaver() 
app = workflow.compile(checkpointer=memory)

/Users/zaidur/anaconda3/envs/lang_env/lib/python3.10/site-packages/langchain_community/utilities/sql_database.py:159: SAWarning: Did not recognize type 'geometry' of column 'geometry'
  self._metadata.reflect(


In [9]:
config = {"configurable": {"thread_id": "react_testing_session_1"}}

print("AIS Maritime ReAct Agent initialized! Type 'exit' or 'quit' to stop.")
print("=" * 60)

while True:
    user_input = input("\nYou: ")
    if user_input.lower() in ['exit', 'quit']:
        print("\nAgent: Shutting down...")
        break
        
    # We now pass the user_input directly into the 'messages' array
    print("-" * 70)
    print("--- User question ---\n", user_input)
    print("-" * 70)
    result = app.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)
    
    # The final message in the state will be the Agent's synthesized answer
    final_message = result['messages'][-1].content
    print(f"\n============ AGENT =============\n {final_message}")

AIS Maritime ReAct Agent initialized! Type 'exit' or 'quit' to stop.
----------------------------------------------------------------------
--- User question ---
 Give me top 50 vessels name with largest lengths
----------------------------------------------------------------------

--- AGENT THINKING ---
Decision: Tool call requested. 
Query:
 WITH latest AS (
  SELECT DISTINCT ON (mmsi) mmsi, vesselname, length
  FROM ais_data
  ORDER BY mmsi, basedatetime DESC
)
SELECT vesselname, length
FROM latest
ORDER BY length DESC
LIMIT 50;
--- VALIDATING SQL ---
--- EXECUTING SQL ---
Query successful. Result length: 1287

--- AGENT THINKING ---
Decision: Gathered enough context. Providing final answer.

============ AGENT =============
 Here are the 50 vessels with the greatest recorded lengths (based on the most recent AIS ping for each ship):

| # | Vessel Name | Length (m) |
|---|-------------|-----------|
| 1 | COOP ENTERPRISE | 489 |
| 2 | E BRONSON INGRAM | 489 |
| 3 | CRIMSON GEM | 487